# Data Analytics Lab Sheet 10: End-to-End Data Analytics Project: Business Intelligence and Predictive Analytics

**Name:** ANURAG PARMAR  
**Roll No.:** 8  
**Section:** B  
**CU ID:** CU24250076  
**Program:** B.Tech CSE  

## Aim
Perform data preparation, EDA, statistical analysis, visualization, churn prediction, and business recommendations in one workflow.

> Run cells from top to bottom. Dataset sources and any synthetic-data notes are documented in the relevant sections.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
sns.set_theme(style="whitegrid")

## 2. Create a bank-customer churn-style dataset

In [ ]:
rng = np.random.default_rng(42)
n = 1500
age = rng.integers(18, 76, n)
tenure = rng.integers(1, 121, n)
balance = np.round(rng.gamma(2.5, 18000, n), 2)
estimated_salary = np.round(rng.uniform(15000, 180000, n), 2)
credit_score = rng.integers(350, 851, n)
products = rng.integers(1, 5, n)
active_member = rng.integers(0, 2, n)
has_card = rng.integers(0, 2, n)
age_group = np.where(age < 30, "Young", np.where(age < 55, "Middle", "Senior"))
risk = (-1.8 + .035 * (age - 40) - .012 * tenure + .000004 * balance
        - .55 * active_member + .28 * (products == 1)
        + .20 * (credit_score < 500) + .20 * (has_card == 0))
prob = 1 / (1 + np.exp(-risk))
churn = rng.binomial(1, np.clip(prob, .03, .90))
df = pd.DataFrame({
    "Age": age, "Tenure": tenure, "Balance": balance,
    "EstimatedSalary": estimated_salary, "CreditScore": credit_score,
    "NumOfProducts": products, "IsActiveMember": active_member,
    "HasCard": has_card, "AgeGroup": age_group, "Churn": churn
})
df["ChurnLabel"] = df["Churn"].map({0: "No", 1: "Yes"})
print("Dataset shape:", df.shape)
display(df.head())

## 3. Data quality and preprocessing

In [ ]:
print("Data types:")
print(df.dtypes)
print("Missing values:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
display(df.describe(include="all").T)

## 4. Exploratory data analysis and KPIs

In [ ]:
display(df["ChurnLabel"].value_counts())
print("Churn rate (%):", round(df["Churn"].mean() * 100, 2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="ChurnLabel", ax=axes[0])
sns.boxplot(data=df, x="ChurnLabel", y="Age", ax=axes[1])
plt.tight_layout()
plt.show()

numeric_cols = ["Age", "Tenure", "Balance", "EstimatedSalary", "CreditScore", "NumOfProducts"]
plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_cols + ["Churn"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

kpis = pd.DataFrame({
    "KPI": ["Customers", "Churned customers", "Churn rate (%)", "Average balance"],
    "Value": [len(df), int(df["Churn"].sum()), df["Churn"].mean() * 100, df["Balance"].mean()]
})
display(kpis.round(2))

## 5. Statistical analysis

In [ ]:
active_churn = df.loc[df["IsActiveMember"] == 1, "Churn"]
inactive_churn = df.loc[df["IsActiveMember"] == 0, "Churn"]
t_stat, p_value = ttest_ind(active_churn, inactive_churn, equal_var=False)
print("Active vs inactive churn t-test:", round(t_stat, 4), "p-value:", round(p_value, 6))

contingency = pd.crosstab(df["AgeGroup"], df["ChurnLabel"])
chi2, chi_p, dof, expected = chi2_contingency(contingency)
print("Age group vs churn chi-square:", round(chi2, 4), "p-value:", round(chi_p, 6))
display(contingency)

## 6. Train a Decision Tree churn model

In [ ]:
features = ["Age", "Tenure", "Balance", "EstimatedSalary", "CreditScore",
            "NumOfProducts", "IsActiveMember", "HasCard"]
X = df[features]
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=42, stratify=y
)
model = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0)
    ]
})
display(metrics)
print(classification_report(y_test, y_pred, zero_division=0))

## 7. Model visualization and feature importance

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

importance = pd.DataFrame({"Feature": features, "Importance": model.feature_importances_})
importance = importance.sort_values("Importance", ascending=False)
display(importance)
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

plt.figure(figsize=(18, 9))
plot_tree(model, feature_names=features, class_names=["No Churn", "Churn"],
          filled=True, rounded=True, max_depth=3, fontsize=8)
plt.show()

## 8. Business insights and export

In [ ]:
print("BUSINESS RECOMMENDATIONS")
print("1. Use predicted churn risk to prioritize customer retention outreach.")
print("2. Investigate customer activity, tenure, and product usage alongside churn outcomes.")
print("3. Compare precision and recall before choosing an operational decision threshold.")
print("4. Validate these findings on a real, representative customer dataset before deployment.")
print("This notebook uses synthetic data; findings are illustrative, not real bank performance.")
df.to_csv("bank_customer_churn_analysis.csv", index=False)
print("Saved bank_customer_churn_analysis.csv")